<a href="https://colab.research.google.com/github/thaisNY/Soul-Code/blob/main/soulcode_web_scraping_dados_e_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Web Scraping, Pré-processamento de Dados e Machine Learning

Imagine que você quer ensinar uma criança a reconhecer frutas. Para isso, você mostra fotos, dá exemplos, corrige os erros dela. Com Inteligência Artificial é a mesma lógica: você precisa de **dados** para ensinar o modelo.

Mas de onde vêm esses dados? Muitas vezes, eles estão espalhados pela internet, em sites, tabelas, notícias, listas de preços. O **Web Scraping** é a técnica de coletar esses dados de forma automática.

Neste notebook, você vai aprender toda essa história, do início ao fim:

1. O que é Web Scraping e as regras do jogo
2. HTML e CSS: a estrutura dos sites
3. Selenium e BeautifulSoup: as ferramentas de coleta
4. NumPy, Pandas, Regex e gráficos: organizando os dados
5. Treinando um modelo de IA com esses dados
6. Melhorando o modelo
7. SQL e NoSQL: guardando tudo com segurança

---
> **Dica:** Execute cada célula na ordem. **NÃO PULE ETAPAS.** Cada bloco de código depende do anterior.
>
> Cada seção tem:  Explicação   Demonstração   Desafio Guiado   Exploração Livre


---

# Parte 1 — As Regras do Jogo: robots.txt

##  Explicação

Antes de sair coletando dados, existe uma regra fundamental que todo programador precisa respeitar: o arquivo **robots.txt**.

Pense assim: você está numa biblioteca enorme. Cada livro é uma página da internet. O `robots.txt` é o regulamento da biblioteca bem na porta de entrada. Ele diz **o que pode** ser consultado por robôs automáticos e **o que não pode**.

Esse arquivo fica sempre no endereço raiz de um site:
- `https://www.google.com/robots.txt`
- `https://www.uol.com.br/robots.txt`

Um arquivo robots.txt típico parece assim:

```
User-agent: *
Disallow: /admin/
Disallow: /usuario/privado/
Allow: /noticias/
```

Isso significa: qualquer robô (`*`) **não pode** acessar `/admin/` ou `/usuario/privado/`, mas **pode** acessar `/noticias/`.

**Scraping sem respeitar o robots.txt pode violar os Termos de Uso do site e, em alguns casos, a legislação brasileira (LGPD).**

---

## Sites brasileiros que permitem Web Scraping (com restrições)

| Site | O que tem de útil | Robots.txt |
|---|---|---|
| Portal da Transparência (transparencia.gov.br) | Dados de gastos públicos | Permite acesso geral |
| IBGE (ibge.gov.br) | Dados demográficos | Permite acesso geral |
| dados.gov.br | Datasets abertos | Permite acesso geral |
| Wikipedia em Português | Textos, tabelas, listas | Permite com limitações |

---

##  Demonstração


In [ ]:
# Lendo o robots.txt de um site de forma automatizada
import urllib.robotparser
import requests

def verificar_robots(url_site, url_pagina):
    """
    Verifica se é permitido fazer scraping de uma página específica.
    url_site   = endereço raiz do site (ex: 'https://pt.wikipedia.org')
    url_pagina = página que queremos raspar
    """
    rp = urllib.robotparser.RobotFileParser()
    rp.set_url(url_site + "/robots.txt")
    rp.read()

    pode = rp.can_fetch("*", url_pagina)
    return pode

# Testando com a Wikipedia
site = "https://pt.wikipedia.org"
pagina = "https://pt.wikipedia.org/wiki/Brasil"

resultado = verificar_robots(site, pagina)
print(f"Posso raspar '{pagina}'? {' SIM' if resultado else ' NÃO'}")

# Também vamos mostrar o conteúdo raw do robots.txt
resposta = requests.get(site + "/robots.txt")
print(f"\nPrimeiras linhas do robots.txt da Wikipedia:")
for linha in resposta.text.split("\n")[:20]:
    print(f"  {linha}")


##  Desafio Guiado


In [ ]:
# DESAFIO 1
# Use a função verificar_robots() para checar se podemos raspar:
# a) https://ibge.gov.br/noticias
# b) https://www.google.com/search
# c) https://pt.wikipedia.org/wiki/Python_(linguagem_de_programacao)

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 2
# Leia o robots.txt do IBGE (https://www.ibge.gov.br/robots.txt)
# e imprima as primeiras 30 linhas.
# Quais caminhos são proibidos?

# Escreva seu código abaixo:


**Dica:** Use `requests.get(url).text` para obter o conteúdo de qualquer URL como texto.

##  Exploração Livre

Teste outros sites que você usa no dia a dia:
- `https://www.mercadolivre.com.br/robots.txt`
- `https://www.reddit.com/robots.txt`
- `https://twitter.com/robots.txt`

Observe as diferenças de permissões entre eles!


In [ ]:
# Escreva seu código aqui


---

# Parte 2 — Antes de Tudo: HTML e CSS

##  Explicação

Web Scraping é basicamente ler o código de uma página e extrair informações dela. Por isso, você precisa entender como uma página é construída.

Toda página na internet é feita com **HTML** (a estrutura, os ossos) e **CSS** (o visual, a roupa). Quando você faz scraping, você está lendo o HTML cru.

Cada elemento HTML tem:
- Uma **tag** (como `<p>`, `<h1>`, `<a>`, `<div>`)
- Atributos como `id` e `class` que identificam o elemento
- Um **conteúdo** entre as tags de abertura e fechamento

Quando você usar **BeautifulSoup**, vai buscar elementos justamente por essas tags, ids e classes.

---

##  Demonstração — Lendo HTML com Python


In [ ]:
# Vamos criar um HTML de exemplo e aprender a "enxergar" sua estrutura
from bs4 import BeautifulSoup

html_exemplo = """
<!DOCTYPE html>
<html>
  <head>
    <title>Loja Virtual Exemplo</title>
  </head>
  <body>
    <h1 id="titulo-principal">Bem-vindo à Loja</h1>
    <p class="descricao">Os melhores produtos com o melhor preço.</p>

    <div class="produto" id="prod-001">
      <h2 class="nome-produto">Notebook Gamer</h2>
      <span class="preco">R$ 4.999,00</span>
      <span class="avaliacao"> (4.2)</span>
      <ul class="especificacoes">
        <li>16GB RAM</li>
        <li>RTX 3060</li>
        <li>512GB SSD</li>
      </ul>
      <a href="/produto/notebook-gamer" class="ver-mais">Ver mais detalhes</a>
    </div>

    <div class="produto" id="prod-002">
      <h2 class="nome-produto">Mouse Sem Fio</h2>
      <span class="preco">R$ 149,90</span>
      <span class="avaliacao"> (4.8)</span>
      <ul class="especificacoes">
        <li>Bluetooth 5.0</li>
        <li>Bateria 30 dias</li>
      </ul>
      <a href="/produto/mouse-sem-fio" class="ver-mais">Ver mais detalhes</a>
    </div>
  </body>
</html>
"""

# Criando o objeto soup para analisar
soup = BeautifulSoup(html_exemplo, 'lxml')

# Vejamos a estrutura bonita do HTML
print("=== Estrutura do HTML ===")
print(soup.prettify()[:600])


In [ ]:
# Agora vamos extrair informações específicas

# 1. Título da página
titulo = soup.find('title')
print(f"Título: {titulo.text}")

# 2. Elemento por ID
h1 = soup.find(id='titulo-principal')
print(f"H1: {h1.text}")

# 3. Todos os elementos por classe
produtos = soup.find_all('div', class_='produto')
print(f"\nNúmero de produtos: {len(produtos)}")

# 4. Navegando dentro de um elemento (pai  filho)
print("\nDetalhes de cada produto:")
for produto in produtos:
    nome = produto.find('h2', class_='nome-produto').text
    preco = produto.find('span', class_='preco').text
    link = produto.find('a').get('href')  # .get() pega atributos
    print(f"  - {nome} | {preco} | {link}")

# 5. Listas (elementos aninhados)
print("\nEspecificações do primeiro produto:")
specs = produtos[0].find_all('li')
for spec in specs:
    print(f"  • {spec.text}")


##  Desafio Guiado

Use o `soup` criado acima para responder:


In [ ]:
# DESAFIO 1
# Extraia as avaliações de todos os produtos (a classe é 'avaliacao')
# Resultado esperado: [' (4.2)', ' (4.8)']

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 2
# Encontre o produto com id='prod-002' e imprima todas as suas especificações

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 3
# Crie uma lista de dicionários com nome e preço de cada produto
# Resultado esperado: [{'nome': 'Notebook Gamer', 'preco': 'R$ 4.999,00'}, ...]

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 4
# Extraia todos os links (href) da página e imprima-os
# Dica: soup.find_all('a', href=True)

# Escreva seu código abaixo:


**Dica:** Use `.find()` para um elemento, `.find_all()` para vários, `.text` para o texto, `.get('atributo')` para atributos.

##  Exploração Livre

Modifique o HTML acima adicionando novos produtos e tente extrair os dados com os mesmos métodos.

Perguntas para explorar:
- O que acontece se você procura por uma classe que não existe?
- E se dois produtos tiverem o mesmo ID?
- Consigo extrair texto de elementos aninhados usando `.get_text()`?


In [ ]:
# Escreva seu código aqui

# Dica: experimente
# soup.find_all('span')
# soup.find('div').find_all('li')
# soup.get_text()


---

# Instalando as Bibliotecas Necessárias

É importante instalar essas bibliotecas porque o Python puro não consegue, sozinho, acessar sites, interpretar HTML ou controlar um navegador.

- **requests:** faz a requisição HTTP para buscar o conteúdo da página.
- **beautifulsoup4:** interpreta e organiza o HTML.
- **selenium:** permite automatizar um navegador (clicar, digitar, esperar).
- **webdriver-manager:** gerencia o driver do navegador automaticamente.
- **lxml:** torna o processamento do HTML mais rápido.

Sem instalar essas bibliotecas, o código de scraping não funcionará.


In [ ]:
# Execute esta célula antes de tudo
!pip install requests beautifulsoup4 selenium webdriver-manager lxml -q
print("Bibliotecas instaladas com sucesso!")


---
## 3.1 — Importando as Bibliotecas

Em Python, importar uma biblioteca é como abrir uma caixa de ferramentas. Cada `import` traz um conjunto de funcionalidades prontas para usar.

As bibliotecas **sempre ficam no início do código**. Pense assim: primeiro você separa as ferramentas na mesa, depois começa a trabalhar.


In [ ]:
# Bibliotecas de scraping
import requests                          # Faz requisições HTTP
from bs4 import BeautifulSoup            # Analisa e navega pelo HTML

# Selenium e seus componentes
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

# Bibliotecas de dados
import pandas as pd                      # Manipulação de tabelas
import numpy as np                       # Cálculos numéricos
import re                                # Expressões regulares

# Bibliotecas de gráficos
import matplotlib.pyplot as plt
import seaborn as sns

# Outras
import time
import json
import os
import sqlite3
import urllib.robotparser

print("Todas as bibliotecas importadas com sucesso!")


---
# Parte 3 — As Ferramentas: Selenium e BeautifulSoup

##  Explicação

Pense na internet como uma rede social. O BeautifulSoup é como tirar print do que já apareceu na tela. O Selenium é como usar o celular normalmente: rolar a tela, clicar em posts e esperar novos conteúdos carregarem.

| Situação | Ferramenta ideal |
|---|---|
| Página estática, conteúdo já no HTML | BeautifulSoup |
| Página dinâmica com JavaScript | Selenium |
| Precisa clicar, fazer login, navegar | Selenium |
| Precisa ser rápido e leve | BeautifulSoup |

---

## 3.2 — BeautifulSoup: Lendo Páginas Estáticas

O fluxo com BeautifulSoup tem sempre três passos:
1. Fazer a requisição (`requests.get`) e obter o HTML
2. Criar o objeto `BeautifulSoup` que vai analisar esse HTML
3. Navegar pelo HTML para extrair o que você precisa

---

##  Demonstração: Coletando dados do IBGE


In [ ]:
# EXEMPLO - Lendo uma página com requests e BeautifulSoup
URL = "https://agenciadenoticias.ibge.gov.br/"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# Passo 1: Fazer a requisição
resposta = requests.get(URL, headers=headers, timeout=10)

# Status 200 = sucesso, 404 = não encontrado, 403 = acesso negado
print(f"Status da requisição: {resposta.status_code}")
print(f"Primeiros 500 caracteres do HTML:\n{resposta.text[:500]}")


In [ ]:
# Criando o objeto soup e navegando pelo HTML
soup = BeautifulSoup(resposta.text, 'lxml')

# Título da página
titulo = soup.find('title')
print(f"Título: {titulo.text if titulo else 'não encontrado'}")

# Todos os links
links = soup.find_all('a', href=True)
print(f"\nTotal de links: {len(links)}")

print("\nPrimeiros 5 links:")
for link in links[:5]:
    print(f"  Texto: {link.text.strip()[:40]} | URL: {link.get('href')}")


##  Desafio Guiado — Praticando com o IBGE


In [ ]:
# DESAFIO 1
# Encontre e imprima o primeiro parágrafo (<p>) da página
# Dica: soup.find('p')

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 2
# Conte quantos elementos <h2> existem na página

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 3
# Extraia todas as imagens (<img>) e imprima os atributos src (use .get('src'))

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 4
# Encontre o elemento com a classe que parece ser do menu de navegação
# Dica: inspecione o HTML com soup.prettify() e procure um nav ou header

# Escreva seu código abaixo:


**Dica:** Use `find()` e `find_all()`.

##  Exploração Livre

Teste outras buscas livremente!
```python
soup.find_all('p')
soup.find_all('h1')
soup.find_all('div')
soup.find_all('img')
soup.prettify()[:1000]
```


In [ ]:
# Escreva seu código aqui


---

## Tabelas HTML: Estrutura Hierárquica

##  Explicação

Tabelas HTML possuem estrutura hierárquica:
- `table`  tabela inteira
- `tr`  linhas
- `td`  células com dados
- `th`  cabeçalhos (headers)

A lógica é:
1. Encontrar a tabela
2. Pegar as linhas
3. Navegar pelas células
4. Extrair o texto

Isso é um exemplo clássico de **elementos aninhados** (pai  filho  neto).

##  Demonstração


In [ ]:
# EXEMPLO - Navegando por elementos aninhados (tabela HTML)
html_tabela = """
<html>
<body>
  <table id="dados-populacao">
    <tr>
      <th>Estado</th>
      <th>Populacao</th>
      <th>IDH</th>
      <th>Capital</th>
    </tr>
    <tr>
      <td>São Paulo</td>
      <td>45.919.049</td>
      <td>0,826</td>
      <td>São Paulo</td>
    </tr>
    <tr>
      <td>Minas Gerais</td>
      <td>21.292.666</td>
      <td>0,787</td>
      <td>Belo Horizonte</td>
    </tr>
    <tr>
      <td>Rio de Janeiro</td>
      <td>17.463.349</td>
      <td>0,796</td>
      <td>Rio de Janeiro</td>
    </tr>
    <tr>
      <td>Bahia</td>
      <td>14.873.064</td>
      <td>0,660</td>
      <td>Salvador</td>
    </tr>
  </table>
</body>
</html>
"""

soup_tabela = BeautifulSoup(html_tabela, 'lxml')

# Passo 1: Encontrar a tabela pelo ID
tabela = soup_tabela.find('table', id='dados-populacao')

# Passo 2: Extrair os cabeçalhos (th)
cabecalhos = [th.text for th in tabela.find_all('th')]
print(f"Colunas: {cabecalhos}")

# Passo 3: Extrair as linhas de dados
dados = []
linhas = tabela.find_all('tr')

for linha in linhas[1:]:  # [1:] pula o cabeçalho
    celulas = linha.find_all('td')
    if celulas:  # garante que não é linha vazia
        registro = {cabecalhos[i]: celulas[i].text.strip() for i in range(len(celulas))}
        dados.append(registro)

print("\nDados extraídos:")
for item in dados:
    print(item)

# Passo 4: Transformar em DataFrame
df_estados = pd.DataFrame(dados)
print("\nDataFrame criado:")
print(df_estados)


##  Desafio Guiado — Trabalhando com a tabela


In [ ]:
# DESAFIO 1
# Conte quantas linhas de dados existem (sem contar o cabeçalho)
# Resultado esperado: 4

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 2
# Extraia apenas os nomes dos estados em uma lista
# Resultado esperado: ['São Paulo', 'Minas Gerais', 'Rio de Janeiro', 'Bahia']

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 3
# Encontre o estado com maior IDH
# Dica: converta os valores de IDH para float primeiro (substitua ',' por '.')

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 4
# Adicione uma nova coluna 'Regiao' ao df_estados com os valores corretos
# SP e RJ = Sudeste, MG = Sudeste, BA = Nordeste

# Escreva seu código abaixo:


**Dica:** linhas são `<tr>` e células são `<td>`.

##  Exploração Livre

- Adicione novas linhas na tabela HTML e veja o que acontece
- Crie uma tabela com mais colunas (área, PIB, etc.)
- Tente usar `pd.read_html(html_tabela)` — pandas tem um método nativo para tabelas!


In [ ]:
# Escreva seu código aqui


---

###  O Grande Desafio — BeautifulSoup

Usando o HTML abaixo, extraia todos os produtos e armazene em um DataFrame com as colunas: `nome`, `preco_texto`, `preco_num`, `avaliacao`, `categoria`.

**Dica:** Use `find_all` com a classe correta. Para converter preço, use `re.sub`.


In [ ]:
html_loja = """
<html>
<body>
  <div class="loja">
    <div class="produto" data-categoria="informatica">
      <span class="nome">Notebook Dell</span>
      <span class="preco">R$ 3.299,00</span>
      <span class="stars">4.5</span>
    </div>
    <div class="produto" data-categoria="informatica">
      <span class="nome">Mouse Logitech</span>
      <span class="preco">R$ 89,90</span>
      <span class="stars">4.8</span>
    </div>
    <div class="produto" data-categoria="audio">
      <span class="nome">Fone Sony</span>
      <span class="preco">R$ 549,00</span>
      <span class="stars">4.6</span>
    </div>
    <div class="produto" data-categoria="audio">
      <span class="nome">Caixa JBL</span>
      <span class="preco">R$ 799,00</span>
      <span class="stars">4.3</span>
    </div>
    <div class="produto" data-categoria="informatica">
      <span class="nome">Teclado Mecânico</span>
      <span class="preco">R$ 249,00</span>
      <span class="stars">4.7</span>
    </div>
  </div>
</body>
</html>
"""

soup_loja = BeautifulSoup(html_loja, 'lxml')

# Escreva seu código aqui
# Resultado esperado: DataFrame com 5 linhas e colunas: nome, preco_texto, preco_num, avaliacao, categoria

# Dica: para pegar o atributo data-categoria, use .get('data-categoria')
# Dica: para converter preço: float(re.sub(r'[^\d,]', '', preco).replace(',', '.'))


---

## 3.3 — Selenium: Navegando em Páginas Dinâmicas

##  Explicação

Muitos sites modernos carregam o conteúdo com JavaScript *depois* que a página abre. O BeautifulSoup não vê esse conteúdo porque só lê o HTML inicial. O Selenium resolve isso porque ele controla um navegador de verdade.

No Google Colab, usamos Chrome em modo **headless** (sem interface gráfica).

### O que o código a seguir faz:
- Instala o Google Chrome no Colab
- Configura o Selenium para controlá-lo
- Abre páginas, espera carregar e extrai o HTML resultante

### Instalando o Chrome no Colab


In [ ]:
import subprocess

for cmd in ["chromium-browser --version", "/usr/lib/chromium-browser/chromedriver --version", "/usr/bin/chromedriver --version"]:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"{cmd}")
    print(f"  stdout: {result.stdout.strip()}")
    print(f"  stderr: {result.stderr.strip()}")
    print(f"  código: {result.returncode}")


In [ ]:
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb -q 2>/dev/null || apt-get install -yf -q
!pip install selenium webdriver-manager -q

result = __import__('subprocess').run("google-chrome --version", shell=True, capture_output=True, text=True)
print(f"Chrome instalado: {result.stdout.strip()}")


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

def criar_driver():
    opcoes = Options()
    opcoes.add_argument("--headless=new")
    opcoes.add_argument("--no-sandbox")
    opcoes.add_argument("--disable-dev-shm-usage")
    opcoes.add_argument("--disable-gpu")
    opcoes.add_argument("--window-size=1920,1080")
    opcoes.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)")

    driver_path = ChromeDriverManager().install()
    service = Service(driver_path)
    driver = webdriver.Chrome(service=service, options=opcoes)
    return driver

driver = criar_driver()
print("Driver Selenium iniciado com sucesso!")


##  Demonstração — Abrindo páginas e extraindo dados

O Selenium permite:
1. Abrir páginas dinâmicas
2. Esperar elementos carregarem
3. Interagir com a página (clicar, digitar, scroll)
4. Extrair o HTML renderizado

Fluxo mental: **abrir página  esperar  extrair  interagir  analisar**


In [ ]:
# DEMONSTRAÇÃO 1 - Abrindo a Wikipedia e extraindo o título
try:
    driver.get("https://pt.wikipedia.org/wiki/Brasil")

    wait = WebDriverWait(driver, 10)
    titulo = wait.until(EC.presence_of_element_located((By.ID, "firstHeading")))

    print(f"Título da página: {titulo.text}")

    # Pegando o primeiro parágrafo
    primeiro_paragrafo = driver.find_element(By.CSS_SELECTOR, "#mw-content-text p")
    print(f"\nPrimeiro parágrafo:\n{primeiro_paragrafo.text[:300]}...")

    # Passando HTML para BeautifulSoup para análise mais profunda
    html_pagina = driver.page_source
    soup_wiki = BeautifulSoup(html_pagina, 'lxml')

    secoes = soup_wiki.find_all(['h2', 'h3'])
    print(f"\nSeções encontradas ({len(secoes)}):")
    for sec in secoes[:8]:
        print(f"  {sec.get_text().strip()}")

except Exception as e:
    print(f"Erro: {e}")


In [ ]:
# DEMONSTRACAO 2 - Interacoes com a pagina
try:
    driver.get("https://pt.wikipedia.org/wiki/Intelig%C3%AAncia_artificial")
    time.sleep(2)

    # Seletores disponiveis no Selenium:
    print("Formas de localizar elementos:")
    print("  By.ID           - driver.find_element(By.ID, 'id-do-elemento')")
    print("  By.CLASS_NAME   - driver.find_element(By.CLASS_NAME, 'nome-classe')")
    print("  By.CSS_SELECTOR - driver.find_element(By.CSS_SELECTOR, 'div.classe > p')")
    xpath_ex = "//div[@class='x']"
    print(f"  By.XPATH        - driver.find_element(By.XPATH, '{xpath_ex}')")
    print("  By.NAME         - driver.find_element(By.NAME, 'nome-campo')")

    print(f"\nPagina atual: {driver.title}")

    # Fazendo scroll para baixo
    driver.execute_script("window.scrollTo(0, 1000)")
    time.sleep(1)
    print("Scroll feito com sucesso!")

    # Todos os links da pagina
    links = driver.find_elements(By.TAG_NAME, "a")
    print(f"Total de links na pagina: {len(links)}")

except Exception as e:
    print(f"Erro: {e}")


##  Desafio Guiado — Selenium


In [ ]:
# DESAFIO 1
# Abra a Wikipedia sobre Python e extraia o título principal
# URL: https://pt.wikipedia.org/wiki/Python_(linguagem_de_programação)

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 2
# Extraia os primeiros 3 parágrafos da página

# Escreva seu código abaixo:
# Dica: driver.find_elements(By.CSS_SELECTOR, "#mw-content-text p")[:3]


In [ ]:
# DESAFIO 3
# Conte quantas seções (h2) existem na página de Python

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 4
# Realize scroll até o final da página
# Dica: driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 5
# Extraia o HTML com driver.page_source e use BeautifulSoup
# para pegar todos os links que contêm a palavra "python" no href

# Escreva seu código abaixo:


**Dica:** use `driver.page_source` + BeautifulSoup para análises mais profundas.

##  Exploração Livre

Teste interações livres:
```python
driver.find_elements(By.TAG_NAME, "a")
driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
driver.back()
driver.forward()
driver.refresh()
driver.current_url
```


In [ ]:
# Escreva o seu código aqui


###  O Grande Desafio — Selenium

Usando Selenium, acesse `https://pt.wikipedia.org/wiki/Python_(linguagem_de_programacao)` e colete:

a) O título principal da página (elemento com id `firstHeading`)

b) Todos os links da página que contenham a palavra `"linguagem"` no texto do link

c) O número total de imagens na página

**Dica:** Para filtrar os links, use um loop com `if 'linguagem' in link.text.lower()`


In [ ]:
# Escreva seu código aqui
driver = criar_driver()

# a) Título principal
# Dica: Use driver.get() para abrir a página
# Dica: Use WebDriverWait para esperar o firstHeading

# b) Links com "linguagem"
# Dica: Use driver.find_elements(By.TAG_NAME, "a")
# Dica: Filtre com if 'linguagem' in link.text.lower()

# c) Total de imagens

# Não esqueça de fechar o driver no final!
# driver.quit()


In [ ]:
# Sempre feche o driver ao terminar para liberar memória
try:
    driver.quit()
    print("Driver encerrado.")
except:
    print("Driver já estava encerrado.")


---

# Parte 4 — Organizando os Dados: NumPy, Pandas, Regex e Gráficos

Coletar os dados é só o primeiro passo. Dados brutos da internet são como um caderno de anotações bagunçado: tudo misturado, com erros, espaços desnecessários e formatos inconsistentes.

Agora vamos aprender a **limpar, organizar e visualizar** esses dados.

---

## 4.1 — Regex: Limpeza de Texto

##  Explicação

**Regex (Expressões Regulares)** é uma linguagem para encontrar e manipular padrões dentro de texto. Pense nela como um bisturi para textos.

| Padrão | Significado |
|---|---|
| `\d` | Qualquer dígito (0-9) |
| `\D` | Qualquer não dígito |
| `\w` | Letra, número ou underscore |
| `\s` | Espaço em branco |
| `+` | Um ou mais do anterior |
| `*` | Zero ou mais do anterior |
| `[]` | Conjunto de caracteres |
| `^` | Início da string |
| `$` | Fim da string |

**Os dois métodos mais usados:**
- `re.sub(padrão, substituto, texto)`  substitui ocorrências
- `re.findall(padrão, texto)`  encontra todas as ocorrências

##  Demonstração — Os padrões mais usados no scraping


In [ ]:
# DEMONSTRAÇÃO — Casos reais de limpeza com Regex

print("=== 1. Remover espaços extras ===")
texto = "   São Paulo   "
print(re.sub(r"^\s+|\s+$", "", texto))  # equivale a .strip()

print("\n=== 2. Remover espaços duplicados no meio ===")
texto = "Rio    de     Janeiro"
print(re.sub(r"\s+", " ", texto))

print("\n=== 3. Extrair só os números ===")
valor = "R$ 1.234,56"
print(re.sub(r"\D", "", valor))  # só dígitos

print("\n=== 4. Limpar preço (mantendo vírgula decimal) ===")
valor = "R$ 1.234,56"
limpo = re.sub(r"[^\d,]", "", valor)
print(limpo)  # '1234,56'

print("\n=== 5. Extrair número decimal ===")
texto = "IDH: 0,826"
idh = re.findall(r"\d+,\d+", texto)
print(idh[0])

print("\n=== 6. Extrair e-mails ===")
texto = "Contatos: joao@email.com, maria@gmail.com"
emails = re.findall(r"\w+@\w+\.\w+", texto)
print(emails)

print("\n=== 7. Extrair telefone ===")
texto = "Telefone: (11) 91234-5678"
tel = re.findall(r"\(\d{2}\)\s?\d{4,5}-\d{4}", texto)
print(tel[0])

print("\n=== 8. Remover pontuação ===")
texto = "Produto #123 - Edição Especial!"
print(re.sub(r"[^\w\s]", "", texto))


##  Desafio Guiado — Regex


In [ ]:
# DESAFIO 1
# Extraia o número 45.000.000 do texto abaixo como um inteiro Python
# (remova os pontos e converta para int)
texto = "O Brasil tem aproximadamente 45.000.000 de habitantes urbanos."

# Escreva seu código abaixo:
# Esperado: 45000000 (int)


In [ ]:
# DESAFIO 2
# Limpe e converta o preço abaixo para float
# "R$ 2.499,90"  2499.90
preco_texto = "R$ 2.499,90"

# Escreva seu código abaixo:
# Dica: remova tudo que não é dígito ou vírgula, depois substitua ',' por '.'


In [ ]:
# DESAFIO 3
# Extraia TODOS os preços que aparecem no texto abaixo
texto = "O notebook custa R$ 3.299,00, o mouse R$ 89,90 e o teclado R$ 149,00"

# Escreva seu código abaixo:
# Dica: use re.findall com um padrão que capture R$ seguido de números


In [ ]:
# DESAFIO 4
# Padronize os CEPs abaixo para o formato "XXXXX-XXX"
# Alguns estão sem traço, outros com espaço
ceps = ["01310100", "20040-020", "30112 000", "04538133"]

# Escreva seu código abaixo:
# Dica: primeiro extraia só os dígitos, depois reformate


**Dica:** `re.sub(r'\D', '', texto)` remove tudo que não é número.

##  Exploração Livre

Experimente padrões mais avançados:
```python
re.findall(r'\d{4}', texto)         # 4 dígitos seguidos (anos, CEPs)
re.findall(r'[A-Z][a-z]+', texto)    # palavras com inicial maiúscula
re.sub(r'(\w+)', r'[\1]', texto)    # colocar colchetes em cada palavra
```

Tente criar um regex que capture datas no formato DD/MM/AAAA!


In [ ]:
# Escreva seu código aqui


---
## 4.2 — Pandas: A Planilha do Python

##  Explicação

O **Pandas** trabalha com **DataFrames** — basicamente tabelas. É a ferramenta mais usada para manipulação de dados em Python.

Depois do scraping, os dados vêm sujos:
- preços como texto (`"R$ 3.299,00"`)
- números misturados com palavras (`"4.5 estrelas"`)
- valores vazios
- separadores inconsistentes

O Pandas permite:
* criar tabelas
* limpar  
* transformar  
* filtrar  
* ordenar  
* analisar

Fluxo mental: **dados brutos  DataFrame  limpeza  transformação  análise**

##  Demonstração


In [ ]:
# Simulando dados que vieram de um scraping
dados_brutos = [
    {'produto': 'Notebook Dell', 'preco': 'R$ 3.299,00', 'avaliacao': '4.5 estrelas', 'vendidos': '1.250'},
    {'produto': 'Notebook Lenovo', 'preco': 'R$ 2.799,00', 'avaliacao': '4.2 estrelas', 'vendidos': '980'},
    {'produto': 'Notebook Acer', 'preco': 'R$ 2.499,00', 'avaliacao': '4.0 estrelas', 'vendidos': '2.100'},
    {'produto': 'Notebook Samsung', 'preco': 'R$ 4.199,00', 'avaliacao': '4.7 estrelas', 'vendidos': '750'},
    {'produto': 'Notebook HP', 'preco': 'R$ 1.999,00', 'avaliacao': '3.8 estrelas', 'vendidos': '3.400'},
    {'produto': 'Notebook Asus', 'preco': '', 'avaliacao': '4.3 estrelas', 'vendidos': '560'},
]

df = pd.DataFrame(dados_brutos)
print("DataFrame original (dados sujos):")
print(df)
print(f"\nFormato: {df.shape[0]} linhas x {df.shape[1]} colunas")
print(f"\nTipos de dados:")
print(df.dtypes)


In [ ]:
# Limpando os dados
df_limpo = df.copy()

# 1. Tratando valores vazios
df_limpo['preco'] = df_limpo['preco'].replace('', np.nan)
df_limpo = df_limpo.dropna(subset=['preco'])
print(f"Após remover vazios: {len(df_limpo)} linhas")

# 2. Convertendo preço para número
df_limpo['preco_num'] = df_limpo['preco'].apply(
    lambda x: float(re.sub(r'[^\d,]', '', x).replace(',', '.'))
)

# 3. Extraindo número da avaliação
df_limpo['avaliacao_num'] = df_limpo['avaliacao'].apply(
    lambda x: float(re.search(r'[\d.]+', x).group())
)

# 4. Convertendo vendidos para inteiro
df_limpo['vendidos_num'] = df_limpo['vendidos'].apply(
    lambda x: int(re.sub(r'\.', '', x))
)

# 5. Removendo colunas sujas originais
df_final = df_limpo.drop(columns=['preco', 'avaliacao', 'vendidos'])

print("\nDataFrame limpo (pronto para análise):")
print(df_final)
print(f"\nTipos de dados agora:")
print(df_final.dtypes)


In [ ]:
# Operações essenciais com Pandas
print("=== Filtragem ===")
baratos = df_final[df_final['preco_num'] < 3000]
print(baratos[['produto', 'preco_num']])

print("\n=== Ordenação ===")
mais_vendidos = df_final.sort_values('vendidos_num', ascending=False)
print(mais_vendidos[['produto', 'vendidos_num']])

print("\n=== Estatísticas básicas ===")
print(df_final[['preco_num', 'avaliacao_num', 'vendidos_num']].describe())

print("\n=== Nova coluna calculada ===")
df_final['score'] = df_final['avaliacao_num'] * np.log1p(df_final['vendidos_num'])
print(df_final[['produto', 'score']].sort_values('score', ascending=False))


##  Desafio Guiado — Pandas


In [ ]:
# DESAFIO 1
# Calcule o preço médio dos notebooks

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 2
# Descubra qual notebook tem a maior avaliação
# Dica: df_final.loc[df_final['avaliacao_num'].idxmax()]

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 3
# Liste produtos com mais de 1500 vendidos ordenados do mais vendido para o menos

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 4
# Crie uma coluna "custo_beneficio" = avaliacao_num / (preco_num / 1000)
# Isso representa "estrelas por cada R$1000 gasto"

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 5
# Crie uma coluna "categoria_preco" com:
# 'barato' se preco_num < 2500
# 'médio' se preco_num entre 2500 e 3500
# 'caro' se preco_num > 3500
# Dica: use pd.cut() ou np.where()

# Escreva seu código abaixo:


**Dica:** use `.mean()`, `.max()`, `.sort_values()`, `pd.cut()` e filtragem booleana.

##  Exploração Livre

Experimente criar:
- filtros diferentes
- novas métricas
- rankings personalizados

```python
df_final.head()
df_final.info()
df_final.describe()
df_final.groupby('categoria_preco').mean()
df_final.corr(numeric_only=True)
```


In [ ]:
# Escreva seu código aqui


---
## 4.3 — NumPy: A Força dos Cálculos

##  Explicação

O **NumPy** trabalha com arrays numéricos de forma muito eficiente.
É a base do Pandas e de praticamente todas as bibliotecas de Machine Learning.

**Pandas organiza. NumPy calcula.**

Depois que os dados estão limpos (no Pandas), o NumPy entra para:
- Fazer cálculos estatísticos rápidos
- Normalizar dados (preparar para ML)
- Calcular correlações
- Fazer operações vetorizadas

##  Demonstração


In [ ]:
# Convertendo colunas do DataFrame em arrays NumPy
precos = np.array(df_final['preco_num'].values)
avaliacoes = np.array(df_final['avaliacao_num'].values)
vendidos = np.array(df_final['vendidos_num'].values)

print("=== Estatísticas básicas ===")
print(f"Preço médio:    R$ {np.mean(precos):.2f}")
print(f"Preço mediano:  R$ {np.median(precos):.2f}")
print(f"Desvio padrão:  R$ {np.std(precos):.2f}")
print(f"Preço mínimo:   R$ {np.min(precos):.2f}")
print(f"Preço máximo:   R$ {np.max(precos):.2f}")

print("\n=== Normalização (0 a 1) — muito usada em ML ===")
precos_norm = (precos - np.min(precos)) / (np.max(precos) - np.min(precos))
print(f"Preços normalizados: {np.round(precos_norm, 3)}")

print("\n=== Correlação entre variáveis ===")
correlacao_pv = np.corrcoef(precos, vendidos)[0, 1]
correlacao_av = np.corrcoef(avaliacoes, vendidos)[0, 1]
print(f"Correlação preço x vendas:     {correlacao_pv:.3f}")
print(f"Correlação avaliação x vendas: {correlacao_av:.3f}")
print("(1.0 = positiva perfeita, -1.0 = negativa perfeita, 0 = sem correlação)")

print("\n=== Percentis ===")
print(f"Q1 (25%): R$ {np.percentile(precos, 25):.2f}")
print(f"Q2 (50%): R$ {np.percentile(precos, 50):.2f}")
print(f"Q3 (75%): R$ {np.percentile(precos, 75):.2f}")


##  Desafio Guiado — NumPy


In [ ]:
# DESAFIO 1
# Calcule a média de vendidos usando NumPy

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 2
# Encontre o ÍNDICE do produto com maior preço usando np.argmax()
# Depois use esse índice para descobrir o NOME do produto

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 3
# Normalize a coluna de vendidos (0 a 1) usando a fórmula:
# (valor - minimo) / (maximo - minimo)

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 4
# Calcule o coeficiente de variação dos preços
# Fórmula: desvio_padrao / media
# Interprete: quanto menor, mais uniformes são os preços

# Escreva seu código abaixo:


**Dica:** use `np.argmax()`, `np.mean()`, `np.std()`, `np.corrcoef()`.

##  Exploração Livre

```python
np.sum(precos)
np.var(precos)
np.log(precos)         # transformação logarítmica
np.sqrt(precos)        # raiz quadrada
np.percentile(precos, [10, 25, 50, 75, 90])
```


In [ ]:
# Escreva seu código aqui


---
## 4.4 — Visualizando os Dados com Gráficos

##  Explicação

Ver os dados visualmente ajuda a entender padrões que os números sozinhos não mostram.

Regra prática:
- **Barra:**  comparação entre categorias
- **Linha:**  tendência ao longo do tempo
- **Dispersão (scatter):**  relação entre duas variáveis
- **Histograma:**  distribuição de valores
- **Pizza:**  participação no total

##  Demonstração


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Análise dos Dados Coletados por Web Scraping', fontsize=14, fontweight='bold')

# Gráfico 1: Preço por produto (barras horizontais)
axes[0, 0].barh(df_final['produto'], df_final['preco_num'], color='steelblue')
axes[0, 0].set_title('Preço por Produto')
axes[0, 0].set_xlabel('Preço (R$)')

# Gráfico 2: Avaliação por produto (barras coloridas)
cores = ['green' if a >= 4.5 else 'orange' if a >= 4.0 else 'red' for a in df_final['avaliacao_num']]
axes[0, 1].bar(df_final['produto'], df_final['avaliacao_num'], color=cores)
axes[0, 1].set_title('Avaliação por Produto')
axes[0, 1].set_ylabel('Estrelas')
axes[0, 1].set_ylim(0, 5)
axes[0, 1].axhline(y=4.0, color='red', linestyle='--', alpha=0.5, label='Mínimo 4.0')
axes[0, 1].legend()
plt.setp(axes[0, 1].get_xticklabels(), rotation=45, ha='right')

# Gráfico 3: Dispersão Preço x Vendas
scatter = axes[1, 0].scatter(
    df_final['preco_num'], df_final['vendidos_num'],
    s=df_final['avaliacao_num']*80,
    c=df_final['avaliacao_num'], cmap='RdYlGn', alpha=0.8
)
axes[1, 0].set_title('Preço x Vendas\n(tamanho = avaliação)')
axes[1, 0].set_xlabel('Preço (R$)')
axes[1, 0].set_ylabel('Unidades Vendidas')
plt.colorbar(scatter, ax=axes[1, 0], label='Avaliação')
for i, row in df_final.iterrows():
    nome = row['produto'].replace('Notebook ', '')
    axes[1, 0].annotate(nome, (row['preco_num'], row['vendidos_num']),
                        textcoords='offset points', xytext=(5, 5), fontsize=8)

# Gráfico 4: Histograma dos preços
axes[1, 1].hist(df_final['preco_num'], bins=5, color='purple', edgecolor='white', alpha=0.8)
axes[1, 1].set_title('Distribuição dos Preços')
axes[1, 1].set_xlabel('Preço (R$)')
axes[1, 1].set_ylabel('Frequência')
axes[1, 1].axvline(df_final['preco_num'].mean(), color='red', linestyle='--', label='Média')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('analise_scraping.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico salvo como 'analise_scraping.png'")


##  Desafio Guiado — Gráficos


In [ ]:
# DESAFIO 1
# Crie um gráfico de pizza mostrando a participação de cada produto nas vendas totais
# Dica: plt.pie(valores, labels=nomes, autopct='%1.1f%%')

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 2
# Crie um gráfico de barras mostrando preço x produto,
# mas destaque o produto mais caro com uma cor diferente

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 3
# Crie um gráfico de dispersão entre avaliação e número de vendidos
# Adicione o nome de cada produto como anotação
# O que você observa? Produtos melhor avaliados vendem mais?

# Escreva seu código abaixo:


**Dica:** use `plt.hist()`, `plt.pie()`, `plt.scatter()`, `plt.bar()`.

##  Exploração Livre

```python
sns.pairplot(df_final.select_dtypes(include='number'))
df_final.plot(kind='box')
sns.heatmap(df_final.corr(numeric_only=True), annot=True, cmap='coolwarm')
```


In [ ]:
# Escreva seu código aqui


###  O Grande Desafio — Parte 4

Com o `df_final` criado acima:

a) Use NumPy para calcular a média de vendidos

b) Use Pandas para criar uma nova coluna `categoria_preco` com os valores 'barato' (< R$2500), 'medio' (R$2500-3500) e 'caro' (> R$3500)

c) Crie um gráfico de pizza mostrando a distribuição de produtos por `categoria_preco`

d) **Desafio bônus:** Crie um gráfico que mostre tanto o preço quanto as vendas de cada produto no mesmo gráfico (eixo duplo Y)

**Dica:** Use `pd.cut()` para criar categorias. Para eixo duplo Y, use `ax.twinx()`


In [ ]:
# Escreva seu código aqui

# a) Média de vendidos com NumPy

# b) Coluna categoria_preco

# c) Gráfico de pizza

# d) [Bônus] Gráfico com eixo duplo Y


---

# Parte 5 — A Razão de Tudo: Treinando um Modelo de IA

##  Explicação

Todo esse trabalho de coleta, limpeza e organização de dados tem um propósito maior: **alimentar um modelo de Inteligência Artificial**.

Imagine que você coletou dados de 10.000 notebooks: preços, avaliações e quantidade de vendas. Com esses dados, você pode ensinar um modelo a **prever se um notebook vai vender bem ou mal**.

Esse é o núcleo do **Machine Learning supervisionado**: você mostra exemplos com respostas conhecidas, e o modelo aprende os padrões.

**Conceitos essenciais:**

- **Features (X):** as variáveis de entrada (preço, avaliação, marca)
- **Target (y):** o que queremos prever (vendas, popularidade)
- **Treino/Teste:** dividimos os dados — 80% para treinar, 20% para testar se o modelo aprendeu de verdade

##  Demonstração — Regressão Linear


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# Dataset simulado (como se tivesse vindo de um scraping)
dados = {
    "preco":     [3299, 2799, 2499, 4199, 1999, 3599, 2899, 2299, 3100, 2650],
    "avaliacao": [4.5,  4.2,  4.0,  4.7,  3.8,  4.6,  4.1,  3.9,  4.4, 4.3],
    "vendidos":  [1250, 980,  2100, 750,  3400, 860,  1500, 2700, 1100, 1800]
}

df_ml = pd.DataFrame(dados)

print("Dataset:")
print(df_ml)
print(f"\n{len(df_ml)} produtos")


In [ ]:
# Passo 1: Separar features (X) e target (y)
X = df_ml[["preco", "avaliacao"]]  # entradas
y = df_ml["vendidos"]              # saída que queremos prever

print("Features (X):")
print(X.head())
print(f"\nTarget (y):")
print(y.values)


In [ ]:
# Passo 2: Dividir em treino e teste
# test_size=0.2 significa que 20% vai para teste
# random_state=42 garante reprodutibilidade (sempre mesmos resultados)
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Dados de treino: {len(X_treino)} amostras")
print(f"Dados de teste:  {len(X_teste)} amostras")


In [ ]:
# Passo 3: Treinar o modelo
modelo = LinearRegression()
modelo.fit(X_treino, y_treino)  # aqui o modelo "aprende"

print("Modelo treinado!")
print(f"Coeficientes: {dict(zip(X.columns, modelo.coef_))}")
print(f"Intercepto: {modelo.intercept_:.2f}")
print("\nInterpretação: para cada R$1 a mais no preço, as vendas mudam em {:.4f} unidades".format(modelo.coef_[0]))


In [ ]:
# Passo 4: Fazer previsões
previsoes = modelo.predict(X_teste)

print("Comparação previsão x real:")
for i, (prev, real) in enumerate(zip(previsoes, y_teste.values)):
    erro = abs(prev - real)
    print(f"  Produto {i+1}: previsto={prev:.0f} | real={real} | erro={erro:.0f}")

# Passo 5: Avaliar o modelo
mae = mean_absolute_error(y_teste, previsoes)
r2 = r2_score(y_teste, previsoes)

print(f"\n=== Métricas de Avaliação ===")
print(f"MAE (Erro Médio Absoluto): {mae:.1f} unidades")
print(f"R² (Coeficiente de Determinação): {r2:.3f}")
print(f"\nInterpretação do R²:")
print(f"  R² = 1.0  perfeito | R² = 0.0  aleatório | R² < 0  pior que aleatório")
print(f"  Nosso modelo explica {r2*100:.1f}% da variação nas vendas")


In [ ]:
# Visualizando: previsão vs real
plt.figure(figsize=(8, 5))
plt.scatter(y_teste, previsoes, color='blue', s=100, label='Previsões')
plt.plot([min(y_teste), max(y_teste)], [min(y_teste), max(y_teste)],
         'r--', label='Linha perfeita (previsto = real)')
plt.xlabel('Valores Reais (vendidos)')
plt.ylabel('Valores Previstos')
plt.title('Previsão vs Real — Regressão Linear')
plt.legend()
plt.tight_layout()
plt.show()


##  Desafio Guiado — Machine Learning


In [ ]:
# DESAFIO 1
# Adicione uma nova feature ao modelo: 'score' = avaliacao * log(1 + vendidos)
# Treine novamente e compare o R²

df_ml["score"] = df_ml["avaliacao"] * np.log1p(df_ml["vendidos"])

# Escreva seu código abaixo (treine com X = ["preco", "avaliacao", "score"]):


In [ ]:
# DESAFIO 2
# Use o modelo para prever as vendas de um notebook novo:
# preço = R$3000, avaliação = 4.4

# Escreva seu código abaixo:
# Dica: novo = np.array([[3000, 4.4]])
# Depois: modelo.predict(novo)


In [ ]:
# DESAFIO 3
# Transforme o problema em CLASSIFICAÇÃO:
# Crie uma coluna "vende_bem" = 1 se vendidos > 1500, senão 0
# Treine uma Regressão Logística (LogisticRegression)
# E avalie com accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Escreva seu código abaixo:


**Dica:** Para classificação use `LogisticRegression()`. Para regressão use `LinearRegression()`.

##  Exploração Livre

Experimente outros algoritmos:
```python
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
```

Substitua `LinearRegression()` por qualquer um desses e compare os R²!


In [ ]:
# Escreva seu código aqui


---

# Parte 6 — Melhorando o Modelo

##  Explicação

Um primeiro modelo raramente é o melhor possível. A ciência de dados é um processo iterativo: você treina, avalia, ajusta e treina de novo.

Duas técnicas principais:
1. **Feature Engineering** — criar variáveis melhores a partir das existentes
2. **Hyperparameter Tuning** — encontrar as melhores configurações do algoritmo

---

## 6.1 — Feature Engineering: Criando Melhores Variáveis

##  Demonstração


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Criando um dataset para classificacao
# Adicionamos bastante ruido para o modelo nao ficar perfeito
# (um modelo com 100% de acuracia nao aprende — memoriza!)
np.random.seed(7)
n = 150

df_modelo = pd.DataFrame({
    'preco':       np.random.randint(1500, 5000, n),
    'avaliacao':   np.round(np.random.uniform(2.5, 5.0, n), 1),
    'num_reviews': np.random.randint(5, 2000, n),
    'marca':       np.random.choice([0, 1, 2], n)
})

# Target com bastante ruido para criar um problema realista
df_modelo['vendidos'] = (
    4000
    - df_modelo['preco'] * 0.5
    + df_modelo['avaliacao'] * 300
    + df_modelo['num_reviews'] * 0.2
    + np.random.normal(0, 800, n)   # ruido alto = problema mais dificil
)
df_modelo['popular'] = (df_modelo['vendidos'] > 2500).astype(int)

print(f'Dataset: {len(df_modelo)} produtos')
print(f'Populares:     {df_modelo["popular"].sum()} ({df_modelo["popular"].mean()*100:.0f}%)')
print(f'Nao populares: {(df_modelo["popular"]==0).sum()} ({(1-df_modelo["popular"].mean())*100:.0f}%)')
print('\n(Ambas as classes precisam aparecer no treino e no teste!)')
print(df_modelo.head())


In [ ]:
# MODELO BASELINE (sem feature engineering)
X_base = df_modelo[['preco', 'avaliacao', 'num_reviews', 'marca']]
y = df_modelo['popular']

X_treino, X_teste, y_treino, y_teste = train_test_split(X_base, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_treino_s = scaler.fit_transform(X_treino)
X_teste_s = scaler.transform(X_teste)

modelo_base = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_base.fit(X_treino_s, y_treino)
y_pred_base = modelo_base.predict(X_teste_s)
acuracia = accuracy_score(y_teste, y_pred_base)
print(f"Acurácia modelo base: {acuracia*100:.1f}%")


In [ ]:
# MODELO MELHORADO (com feature engineering)
df_melhorado = df_modelo.copy()

# Feature 1: Score de custo-benefício
df_melhorado['custo_beneficio'] = df_melhorado['avaliacao'] / (df_melhorado['preco'] / 1000)

# Feature 2: Confiabilidade (mais reviews = mais confiável)
df_melhorado['confiabilidade'] = np.log1p(df_melhorado['num_reviews'])

# Feature 3: Produto premium (preço alto + marca premium)
df_melhorado['premium'] = ((df_melhorado['preco'] > 4000) & (df_melhorado['marca'] == 2)).astype(int)

# Retreinando com mais features
X2 = df_melhorado[['preco', 'avaliacao', 'num_reviews', 'marca', 'custo_beneficio', 'confiabilidade', 'premium']]
y2 = df_melhorado['popular']

X2_treino, X2_teste, y2_treino, y2_teste = train_test_split(X2, y2, test_size=0.2, random_state=42)

scaler2 = StandardScaler()
X2_treino_s = scaler2.fit_transform(X2_treino)
X2_teste_s = scaler2.transform(X2_teste)

modelo2 = RandomForestClassifier(n_estimators=100, random_state=42)
modelo2.fit(X2_treino_s, y2_treino)
y2_pred = modelo2.predict(X2_teste_s)
acuracia2 = accuracy_score(y2_teste, y2_pred)

print(f"Acurácia modelo base:     {acuracia*100:.1f}%")
print(f"Acurácia modelo melhorado: {acuracia2*100:.1f}%")
print(f"Melhora: {(acuracia2 - acuracia)*100:.1f} pontos percentuais")


---
## 6.2 — Hyperparameter Tuning: Ajustando o Modelo

##  Demonstração


In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score

# GridSearchCV testa automaticamente combinações de parâmetros
parametros = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    parametros,
    cv=5,             # validação cruzada com 5 folds
    scoring='accuracy',
    n_jobs=-1,        # usa todos os núcleos do processador
    verbose=1
)

print("Iniciando busca pelos melhores hiperparâmetros...")
grid_search.fit(X2_treino_s, y2_treino)

print(f"\nMelhores parâmetros: {grid_search.best_params_}")
print(f"Melhor acurácia (cv): {grid_search.best_score_*100:.1f}%")

melhor_modelo = grid_search.best_estimator_
y_final = melhor_modelo.predict(X2_teste_s)
acuracia_final = accuracy_score(y2_teste, y_final)
print(f"Acurácia no teste:   {acuracia_final*100:.1f}%")


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Validacao cruzada
scores_cv = cross_val_score(melhor_modelo, X2_treino_s, y2_treino, cv=5)
print(f'Scores por fold: {np.round(scores_cv, 3)}')
print(f'Media: {scores_cv.mean():.3f} | Desvio: {scores_cv.std():.3f}')
print('Um desvio baixo indica que o modelo e estavel')
print('Desvio = 0.000 pode indicar dataset muito facil (overfitting)')

# Matriz de confusao — com labels explicitos para evitar erro quando
# uma classe nao aparece no conjunto de teste
labels = [0, 1]
label_names = ['Pouco popular', 'Popular']

cm = confusion_matrix(y2_teste, y_final, labels=labels)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Matriz de Confusao do Modelo Final')
plt.tight_layout()
plt.show()

# Resumo manual para entender os erros
vn, fp, fn, vp = cm.ravel() if cm.shape == (2,2) else (cm[0,0], 0, 0, cm[1,1])
print(f'\nVerdadeiros Negativos (acertou Pouco popular): {vn}')
print(f'Falsos Positivos (disse Popular, era Pouco):   {fp}')
print(f'Falsos Negativos (disse Pouco, era Popular):   {fn}')
print(f'Verdadeiros Positivos (acertou Popular):       {vp}')
print('\nDiagonal principal = acertos | Fora da diagonal = erros')


##  Desafio Guiado — Melhorando o Modelo


In [ ]:
# DESAFIO 1
# Crie uma nova feature chamada 'preco_por_estrela' = preco / avaliacao
# Adicione ao dataset e retreine o RandomForest
# Compare a acurácia com o modelo melhorado anterior

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 2
# Use cross_val_score para avaliar a estabilidade do novo modelo
# O desvio padrão caiu ou aumentou?

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 3
# Visualize a importância das features usando feature_importances_
# Dica: modelo.feature_importances_ retorna a importância de cada feature
# Crie um gráfico de barras horizontais com as importâncias

# Escreva seu código abaixo:


##  Exploração Livre

Experimente:
- Outros algoritmos: `GradientBoostingClassifier`, `SVC`
- Normalização Z-score vs MinMax
- Diferentes splits (70/30, 90/10)
- Comparar Regressão Logística vs Random Forest


In [ ]:
# Escreva seu código aqui


---

# Visualizando o Processo de Treino

##  Explicação

Treinar um modelo significa:
1. Fazer previsões
2. Comparar com os valores reais
3. Calcular o erro
4. Ajustar os parâmetros (pesos)
5. Repetir várias vezes (épocas)

Usamos `SGDRegressor` que aprende **passo a passo** (gradiente estocástico), permitindo visualizar como o erro diminui a cada época.

##  Demonstração


In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error

# Criando dataset simples com 1 variável (fácil de visualizar)
np.random.seed(42)
precos_treino = np.linspace(2000, 4000, 50)
vendidos_treino = 5000 - precos_treino + np.random.normal(0, 300, 50)

df_vis = pd.DataFrame({"preco": precos_treino, "vendidos": vendidos_treino})

X_vis = df_vis[["preco"]]
y_vis = df_vis["vendidos"]

X_v_treino, X_v_teste, y_v_treino, y_v_teste = train_test_split(
    X_vis, y_vis, test_size=0.2, random_state=42
)

# Visualizando os dados
plt.figure(figsize=(8, 5))
plt.scatter(X_v_treino, y_v_treino, label='Treino', alpha=0.7)
plt.scatter(X_v_teste, y_v_teste, label='Teste', marker='x', s=100)
plt.title("Dados antes do treino")
plt.xlabel("Preço")
plt.ylabel("Vendas")
plt.legend()
plt.show()


In [ ]:
# Treinando com acompanhamento do erro
modelo_sgd = SGDRegressor(max_iter=1, learning_rate="constant", eta0=0.00000001, random_state=42)

erros_treino = []
erros_teste = []

for epoca in range(100):
    modelo_sgd.partial_fit(X_v_treino, y_v_treino)
    erro_treino = mean_squared_error(y_v_treino, modelo_sgd.predict(X_v_treino))
    erro_teste = mean_squared_error(y_v_teste, modelo_sgd.predict(X_v_teste))
    erros_treino.append(erro_treino)
    erros_teste.append(erro_teste)

# Visualizando a evolução do erro
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(erros_treino, label='Treino', color='blue')
axes[0].plot(erros_teste, label='Teste', color='red', linestyle='--')
axes[0].set_title("Evolução do Erro durante o Treino")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Erro (MSE)")
axes[0].legend()
axes[0].set_yscale('log')  # escala log para ver melhor

# Linha de regressão final
axes[1].scatter(X_v_treino, y_v_treino, alpha=0.7, label='Dados treino')
axes[1].plot(X_v_treino, modelo_sgd.predict(X_v_treino), color='red', linewidth=2, label='Linha aprendida')
axes[1].set_title("Linha de Regressão após Treino")
axes[1].set_xlabel("Preço")
axes[1].set_ylabel("Vendas")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Se a curva do erro desce  o modelo está aprendendo! ")


##  Desafio Guiado — Visualizando o Aprendizado


In [ ]:
# DESAFIO 1
# Aumente o número de épocas para 300.
# O erro continua diminuindo ou estabiliza?

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 2
# Aumente a learning rate para 0.000001 (10x maior).
# O que acontece com a curva de erro?

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 3
# Adicione mais "ruído" aos dados:
# vendidos_ruidoso = 5000 - precos_treino + np.random.normal(0, 800, 50)
# Compare as curvas de erro. O modelo aprende pior?

# Escreva seu código abaixo:


##  Exploração Livre

```python
# Comparar SGDRegressor vs LinearRegression
from sklearn.linear_model import LinearRegression
modelo_lr = LinearRegression()
modelo_lr.fit(X_v_treino, y_v_treino)
# qual tem menor erro no teste?

# Adicionar segunda variável (avaliacao)
# Normalizar os dados antes do treino
from sklearn.preprocessing import StandardScaler
```


In [ ]:
# Escreva seu código aqui


###  O Grande Desafio — ML Completo

Monte um pipeline completo:

a) Crie um dataset com pelo menos 50 amostras simuladas com colunas: `preco`, `avaliacao`, `num_reviews`

b) Crie a feature `vende_bem` (1 se vendidos > 2000, senão 0)

c) Treine um `RandomForestClassifier` com validação cruzada

d) Plote a matriz de confusão

e) **Bônus:** use `GridSearchCV` para otimizar os hiperparâmetros

**Dica:** siga o pipeline: dados  features  split  normalizar  treinar  avaliar  otimizar


In [ ]:
# Escreva seu código aqui

# a) Dataset

# b) Target

# c) Treino com cross_val_score

# d) Matriz de confusão

# e) [Bônus] GridSearchCV


---

# Parte 7 — Guardando os Dados: SQL e NoSQL

##  Explicação

Você coletou os dados, limpou, treinou o modelo. Mas e os dados em si, onde ficam guardados?

**SQL:** Organiza dados em tabelas com esquema fixo. Perfeito para dados estruturados.
**NoSQL:** Documentos flexíveis em formato JSON. Ideal para dados variáveis.

| Situação | Melhor opção |
|---|---|
| Dados com estrutura fixa e bem definida | SQL |
| Relacionamentos complexos entre tabelas | SQL |
| Dados variáveis, documentos, JSON | NoSQL |
| Grande volume, precisa escalar muito | NoSQL |
| Dados de redes sociais, logs, IoT | NoSQL |

Pratique SQL: https://www.sqlteaching.com/#!select

##  Demonstração — SQLite


In [ ]:
# DEMONSTRAÇÃO SQLite (banco SQL local)
conn = sqlite3.connect(':memory:')  # ':memory:' = banco na RAM
cursor = conn.cursor()

# Criando a tabela
cursor.execute('''
    CREATE TABLE IF NOT EXISTS produtos_scraping (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nome TEXT NOT NULL,
        preco REAL,
        avaliacao REAL,
        vendidos INTEGER,
        data_coleta TEXT
    )
''')
conn.commit()
print("Tabela criada!")

# Inserindo dados com cursor.execute()
produtos = [
    ('Notebook Dell', 3299.0, 4.5, 1250, '2024-03-15'),
    ('Notebook Lenovo', 2799.0, 4.2, 980, '2024-03-15'),
    ('Notebook Acer', 2499.0, 4.0, 2100, '2024-03-15'),
    ('Notebook Samsung', 4199.0, 4.7, 750, '2024-03-15'),
    ('Notebook HP', 1999.0, 3.8, 3400, '2024-03-15'),
]

cursor.executemany(
    "INSERT INTO produtos_scraping (nome, preco, avaliacao, vendidos, data_coleta) VALUES (?, ?, ?, ?, ?)",
    produtos
)
conn.commit()
print(f"{len(produtos)} registros inseridos!")


In [ ]:
# Consultando com SQL
print("=== Todos os produtos ===")
resultado1 = pd.read_sql_query("SELECT * FROM produtos_scraping", conn)
print(resultado1)

print("\\n=== Produtos com avaliacao >= 4.2, ordenados por preco ===")
query = """
    SELECT nome, preco, avaliacao, vendidos
    FROM produtos_scraping
    WHERE avaliacao >= 4.2
    ORDER BY preco ASC
"""
resultado2 = pd.read_sql_query(query, conn)
print(resultado2)

print("\\n=== Estatisticas agregadas ===")
query_stats = """
    SELECT
        AVG(preco) as preco_medio,
        MAX(avaliacao) as melhor_avaliacao,
        SUM(vendidos) as total_vendidos
    FROM produtos_scraping
"""
resultado3 = pd.read_sql_query(query_stats, conn)
print(resultado3)

conn.close()


In [ ]:
# DEMONSTRAÇÃO NoSQL (estrutura MongoDB-like em Python)
documento_produto = {
    "nome": "Notebook Dell XPS 15",
    "preco": 3299.00,
    "especificacoes": {
        "ram": "16GB",
        "processador": "Intel Core i7",
        "armazenamento": "512GB SSD"
    },
    "avaliacoes": [
        {"usuario": "carlos_tech", "nota": 5, "comentario": "Excelente!", "data": "2024-02-10"},
        {"usuario": "ana_design", "nota": 4, "comentario": "Ótimo, mas bateria poderia melhorar", "data": "2024-02-22"},
    ],
    "tags": ["notebook", "dell", "xps", "workstation"],
    "metadata": {
        "data_scraping": "2024-03-15",
        "fonte": "loja_virtual_xyz",
    }
}

print("Documento no formato NoSQL (MongoDB):")
print(json.dumps(documento_produto, indent=2, ensure_ascii=False))

# Acessando dados aninhados
print(f"\nNome: {documento_produto['nome']}")
print(f"RAM: {documento_produto['especificacoes']['ram']}")
print(f"Número de avaliações: {len(documento_produto['avaliacoes'])}")
print(f"Nota média: {sum(a['nota'] for a in documento_produto['avaliacoes']) / len(documento_produto['avaliacoes']):.1f}")


##  Desafio Guiado — SQL e NoSQL


In [ ]:
# DESAFIO 1
# Crie um banco SQLite em memória com uma tabela de 'usuarios'
# Colunas: id, nome, email, cidade, data_cadastro
# Insira pelo menos 5 usuários manualmente com cursor.execute()

conn2 = sqlite3.connect(':memory:')
cursor2 = conn2.cursor()

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 2
# Com o banco criado acima, escreva queries SQL para:
# a) Listar todos os usuários da cidade "São Paulo"
# b) Contar quantos usuários existem por cidade (GROUP BY)
# c) Encontrar o usuário registrado mais recentemente

# Escreva seu código abaixo:


In [ ]:
# DESAFIO 3
# Crie um dicionário Python representando um documento NoSQL
# de um produto de sua escolha com:
# - nome e preço
# - lista de pelo menos 3 avaliações
# - subdocumento de especificações técnicas
# - lista de tags
# Depois acesse e imprima cada parte do documento

# Escreva seu código abaixo:


**Dica SQL:** `WHERE cidade = 'São Paulo'` | `GROUP BY cidade` | `ORDER BY data_cadastro DESC LIMIT 1`

##  Exploração Livre

Experimente:
```python
# Salvar o DataFrame em um banco SQLite de verdade (arquivo)
conn_arquivo = sqlite3.connect('meu_banco.db')
df_final.to_sql('produtos', conn_arquivo, if_exists='replace', index=False)
conn_arquivo.close()

# JOINs entre tabelas
# UPDATE e DELETE
# Índices para performance
```


In [ ]:
# Escreva seu código aqui


###  O Grande Desafio — SQL

a) Crie um banco com **duas tabelas** relacionadas: `produtos` e `categorias`
   - `categorias`: id, nome, descricao
   - `produtos`: id, nome, preco, avaliacao, vendidos, categoria_id (FK)

b) Insira pelo menos 3 categorias e 8 produtos

c) Escreva um JOIN que retorne: nome do produto, preço e nome da categoria

d) Escreva uma query que retorne a avaliação média por categoria

**Dica:** `JOIN categorias ON produtos.categoria_id = categorias.id`


In [ ]:
# Escreva seu código aqui
import sqlite3

conn_desafio = sqlite3.connect(':memory:')
cursor_desafio = conn_desafio.cursor()

# a) Criar tabelas

# b) Inserir dados

# c) JOIN

# d) AVG por categoria


---

#  Recapitulando a História

Você chegou ao final! Arrasou! Veja o que você viu:

**robots.txt e ética** — Todo trabalho de coleta começa com responsabilidade. Respeitar os limites dos sites não é opcional.

**HTML e CSS** — O mapa do território. Sem entender a estrutura de uma página, você não consegue navegar nela de forma automatizada.

**BeautifulSoup** — Para páginas estáticas: encontre tags, extraia texto, navegue por elementos aninhados.

**Selenium** — Para páginas dinâmicas: abra, aguarde, interaja, extraia.

**Regex, Pandas e NumPy** — A oficina de limpeza. Dados brutos raramente estão prontos para uso.

**Gráficos** — Os dados só contam sua história completa quando são visualizados.

**Machine Learning** — O destino final dos dados. Com dados limpos e organizados, você pode treinar modelos que aprendem padrões.

**Melhoria do modelo** — Feature engineering, hyperparameter tuning e validação cruzada.

**SQL e NoSQL** — A memória persistente de tudo.

---

Esta é a fundação de qualquer pipeline de Inteligência Artificial baseado em dados reais.

**Continue estudando:**
- HTML/CSS: https://www.w3schools.com/html/default.asp
- SQL interativo: https://www.sqlteaching.com/#!select
- MongoDB: https://www.w3schools.com/mongodb/index.php
- Scikit-learn: https://scikit-learn.org/stable/

**Conselho final:** consistência vale mais do que pressa. Um pouco de prática todos os dias faz toda a diferença.
